# GenAI Insights

This notebook demonstrates the lightweight Version 0.1 GenAI insight layer. It reads existing forecast, model performance, segment, and diagnostic CSV outputs, then generates structured report-level insights in JSON and Markdown.

If `OPENAI_API_KEY` is not set, the pipeline uses deterministic rule-based fallback insights so the demo remains fully runnable.

In [1]:
from pathlib import Path
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))

OPENAI_API_KEY loaded: True


## 1. Project Setup

In [2]:
from pathlib import Path
import os
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

FORECASTS_DIR = PROJECT_ROOT / "outputs" / "forecasts"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
SEGMENTS_DIR = PROJECT_ROOT / "outputs" / "segments"
DIAGNOSTICS_DIR = PROJECT_ROOT / "outputs" / "diagnostics"
INSIGHTS_DIR = PROJECT_ROOT / "outputs" / "insights"

for output_dir in [FORECASTS_DIR, METRICS_DIR, SEGMENTS_DIR, DIAGNOSTICS_DIR, INSIGHTS_DIR]:
    output_dir.mkdir(parents=True, exist_ok=True)

## 2. Create Demo Inputs Only If Needed

The production insight generator does not contain sample data. This notebook creates a tiny fallback dataset only when the expected CSV outputs are missing.

In [3]:
def file_exists_any(directory: Path, filenames: list[str]) -> bool:
    return any((directory / filename).exists() for filename in filenames)


missing_forecasts = not file_exists_any(
    FORECASTS_DIR,
    ["report_forecasts.csv", "report_view_forecasts_latest.csv", "sample_baseline_forecasts.csv"],
)
missing_metrics = not file_exists_any(
    METRICS_DIR,
    ["model_performance.csv", "report_view_metrics_latest.csv", "report_model_comparison_latest.csv"],
)
missing_segments = not (SEGMENTS_DIR / "report_segments.csv").exists()
missing_diagnostics = not (DIAGNOSTICS_DIR / "report_diagnostics.csv").exists()

if missing_forecasts:
    pd.DataFrame(
        [
            {"report_id": "R_DEMO_001", "report_name": "Executive Sales", "date": "2026-06-01", "forecast": 42},
            {"report_id": "R_DEMO_001", "report_name": "Executive Sales", "date": "2026-06-02", "forecast": 45},
            {"report_id": "R_DEMO_002", "report_name": "Operations Detail", "date": "2026-06-01", "forecast": 8},
            {"report_id": "R_DEMO_002", "report_name": "Operations Detail", "date": "2026-06-02", "forecast": 6},
        ]
    ).to_csv(FORECASTS_DIR / "report_forecasts.csv", index=False)

if missing_metrics:
    pd.DataFrame(
        [
            {"report_id": "R_DEMO_001", "report_name": "Executive Sales", "selected_model": "seasonal_naive", "selected_mae": 4.2, "selected_wape": 0.18, "forecast_reliable": True},
            {"report_id": "R_DEMO_002", "report_name": "Operations Detail", "selected_model": "naive", "selected_mae": 5.8, "selected_wape": 0.62, "forecast_reliable": False},
        ]
    ).to_csv(METRICS_DIR / "model_performance.csv", index=False)

if missing_segments:
    pd.DataFrame(
        [
            {"report_id": "R_DEMO_001", "report_name": "Executive Sales", "report_segment": "high_value", "segment_reason": "High usage and broad stakeholder reach."},
            {"report_id": "R_DEMO_002", "report_name": "Operations Detail", "report_segment": "at_risk", "segment_reason": "Usage is declining and repeat engagement is weak."},
        ]
    ).to_csv(SEGMENTS_DIR / "report_segments.csv", index=False)

if missing_diagnostics:
    pd.DataFrame(
        [
            {"report_id": "R_DEMO_001", "report_name": "Executive Sales", "report_segment": "high_value", "performance_issue": False, "engagement_issue": False, "dependency_risk": False, "inactive_risk": False, "main_diagnostic": "healthy_or_monitor", "diagnostic_summary": "No major diagnostic rule was triggered; continue monitoring."},
            {"report_id": "R_DEMO_002", "report_name": "Operations Detail", "report_segment": "at_risk", "performance_issue": False, "engagement_issue": True, "dependency_risk": False, "inactive_risk": False, "main_diagnostic": "engagement_issue", "diagnostic_summary": "Usage is declining and repeat usage is relatively low."},
        ]
    ).to_csv(DIAGNOSTICS_DIR / "report_diagnostics.csv", index=False)

{
    "created_forecast_sample": missing_forecasts,
    "created_metrics_sample": missing_metrics,
    "created_segments_sample": missing_segments,
    "created_diagnostics_sample": missing_diagnostics,
}

{'created_forecast_sample': False,
 'created_metrics_sample': False,
 'created_segments_sample': False,
 'created_diagnostics_sample': False}

## 3. Generate Batch Insights

In [5]:
from src.genai.insight_generator import generate_report_insights, save_insights

mode = "openai" if os.getenv("OPENAI_API_KEY") else "rule-based fallback"
print(f"Insight generation mode: {mode}")

insights = generate_report_insights(project_root=PROJECT_ROOT)
output_paths = save_insights(insights, project_root=PROJECT_ROOT)

output_paths

Insight generation mode: openai
Generating insight 1/30: Report_001
Generating insight 2/30: Report_002
Generating insight 3/30: Report_003
Generating insight 4/30: Report_004
Generating insight 5/30: Report_005
Generating insight 6/30: Report_006
Generating insight 7/30: Report_007
Generating insight 8/30: Report_008
Generating insight 9/30: Report_009
Generating insight 10/30: Report_010
Generating insight 11/30: Report_011
Generating insight 12/30: Report_012
Generating insight 13/30: Report_013
Generating insight 14/30: Report_014
Generating insight 15/30: Report_015
Generating insight 16/30: Report_016
Generating insight 17/30: Report_017
Generating insight 18/30: Report_018
Generating insight 19/30: Report_019
Generating insight 20/30: Report_020
Generating insight 21/30: Report_021
Generating insight 22/30: Report_022
Generating insight 23/30: Report_023
Generating insight 24/30: Report_024
Generating insight 25/30: Report_025
Generating insight 26/30: Report_026
Generating insi

{'json': PosixPath('/Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/outputs/insights/report_ai_insights.json'),
 'markdown': PosixPath('/Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/outputs/insights/report_ai_insights.md')}

## 4. Review Structured Output

In [6]:
pd.DataFrame(insights).head()

,health_status,forecast_summary,key_drivers,hypotheses,recommended_actions,confidence,report_id,report_name,generation_mode,api_attempts
0,Healthy,The usage forecast for Report_001 is stable an...,[Specialized usage pattern with consistent rep...,[Stable niche segment usage is driven by a foc...,[Continue regular monitoring to detect any eme...,high,R_001,Report_001,openai,1
1,Healthy,The forecast for Report_002 is reliable and st...,[High average views and unique users placing t...,[Stable user engagement is maintaining forecas...,[Continue regular monitoring to detect any eme...,high,R_002,Report_002,openai,1
2,Healthy,The usage forecast for Report_003 indicates st...,"[Moderate to low usage levels, Stable forecast...",[Usage remains consistent due to steady user e...,[Continue regular monitoring to detect any eme...,high,R_003,Report_003,openai,1
3,Healthy,The usage forecast for Report_004 indicates st...,[Specialized niche segment with consistent rep...,"[The niche nature of the report drives steady,...",[Continue regular monitoring to detect any eme...,high,R_004,Report_004,openai,1
4,Healthy,The usage forecast for Report_005 is stable an...,[High engagement indicated by top quartile ave...,[Sustained high user interest is driving stabl...,[Continue regular monitoring to detect any eme...,high,R_005,Report_005,openai,1


## 5. Output Files

In [ ]:
for label, path in output_paths.items():
    print(f"{label}: {path.relative_to(PROJECT_ROOT)}")